In [5]:
import json
import lightrag
import tiktoken

In [6]:
with open("tiktokenestimates/Datasets/Corpus/novel.json", "r") as f:
    str_ = f.read()
    json_ = json.loads(str_)

### Indexing

In [7]:
novel_text = ""
for idx in json_:
    novel_text += idx["context"]

len(novel_text)

4819610

In [8]:
enc = tiktoken.get_encoding("o200k_base")

In [9]:
novel_tokens = len(enc.encode(novel_text))
novel_tokens

1116794

In [10]:
import math
chunk_size = 1200
overlap = 100
step = chunk_size - overlap
novel_chunks = max(1, math.ceil(novel_tokens / step))
novel_chunks 

1016

In [11]:
# tokens for lightrag prompts
prompt_counts = {k: len(enc.encode(str(v))) for k, v in lightrag.prompt.PROMPTS.items()}
prompt_counts

{'DEFAULT_TUPLE_DELIMITER': 5,
 'DEFAULT_COMPLETION_DELIMITER': 6,
 'entity_extraction_system_prompt': 1110,
 'entity_extraction_user_prompt': 174,
 'entity_continue_extraction_user_prompt': 394,
 'entity_extraction_examples': 1892,
 'summarize_entity_descriptions': 461,
 'fail_response': 16,
 'rag_response': 567,
 'naive_rag_response': 559,
 'kg_query_context': 91,
 'naive_query_context': 61,
 'keywords_extraction': 370,
 'keywords_extraction_examples': 242}

In [12]:
entity_prompt_tokens = prompt_counts["entity_extraction_system_prompt"] + prompt_counts["entity_extraction_examples"] 
entity_prompt_tokens

3002

In [13]:
user_prompt_input_tokens = prompt_counts["entity_extraction_user_prompt"] + prompt_counts["entity_continue_extraction_user_prompt"]
user_prompt_input_tokens

568

In [14]:
input_tokens = novel_tokens + user_prompt_input_tokens
cached_tokens = entity_prompt_tokens * novel_chunks

In [15]:
est_input_cost = input_tokens * (10**-6) * 1.25
est_input_cost

1.3967025

In [16]:
est_cached_cost = cached_tokens * (10**-6) * 0.125
est_cached_cost

0.381254

In [17]:
embedding_tokens = novel_tokens
embedding_cost = embedding_tokens * (10**-6) * 0.002
embedding_cost

0.002233588

### Indexing output tokens

In [19]:
path = "./dickens/kv_store_llm_response_cache.json"
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)
    tokens = 0
    for k,v in data.items():
        tokens += len(enc.encode(v['return']))
print(f"Total output tokens: {tokens}")

Total output tokens: 1047365


### Retrieval Input tokens

In [21]:
cached_prompt_tokens = prompt_counts['keywords_extraction'] + prompt_counts['keywords_extraction_examples']
cached_prompt_tokens

612

In [22]:
# Retrieval question input
path = "tiktokenestimates/datasets/questions/novel_questions.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)
    tokens = 0
    for v in data[5:10]:
        tokens += len(enc.encode(v['question']))
print(f"Total query tokens: {tokens}")

Total query tokens: 135


In [66]:
# query context input

log_path = "./dickens/llm_prompts.log"
results = []
context_blocks = []
collecting = False
tokens = 0

with open(log_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        # print("\n")
        # print(line)
        if "---Context---" in line:
            results.append(line)

for line in results:   
    try:
        start = line.index("---Context---")
        context_blocks.append(line[start:])
    except ValueError:
        continue

tokens += len(enc.encode("\n".join(context_blocks)))
print(f"Total query context tokens: {tokens}")


Total query context tokens: 145283


In [24]:
# query output tokens
path = "./dickens/results/novel_results_hybrid.json"
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)
    tokens = 0
    for record in data:
        tokens += len(enc.encode(record['answer']))
    print(f"Total query output tokens: {tokens}")

Total query output tokens: 675
